In [1]:
import yaml
import cudf
import numba
import pandas as pd
import re
import string
import contractions
from textblob import TextBlob
import emoji
import tqdm
from tqdm import tqdm
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from symspellpy import SymSpell, Verbosity
import pkg_resources

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/grv06sid/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/tmp/ipykernel_21957/2911370152.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
df = cudf.read_csv('datasets/Quora Questions Pair Dataset/train.csv')
#df = pd.read_csv('datasets/Quora Questions Pair Dataset/train.csv')

In [3]:
# remove unnecessary columns
df.drop(columns=['id','qid1','qid2'], inplace=True)

In [4]:
df.sample(5)

,question1,question2,is_duplicate
83171,How improve my english mail writing skills?,How can I improve my English writing skills? W...,1
84640,What is the average IQ?,What is the average IQ of a human?,0
176125,In the film universe of Alien (1979) and Alien...,Aliens (1986 movie): How many years was Ripley...,0
224939,Why do people fall in love with physical appea...,"Is it racist to be attracted more, or less, to...",0
141964,Will Bernie supporters withhold their vote for...,Will Bernie Sanders supporters support Clinton...,0


In [5]:
#lower case
df['question1'] = df['question1'].str.lower()
df['question2'] = df['question2'].str.lower()

In [6]:
# remove html tags
df['question1'] = df['question1'].str.replace(r'<.*?>','',regex=True)
df['question2'] = df['question2'].str.replace(r'<.*?>','',regex=True)

In [7]:
# remove web links
df['question1'] = df['question1'].str.replace(r'http\S+|www\.\S+','',regex=True)
df['question2'] = df['question2'].str.replace(r'http\S+|www\.\S+','',regex=True)

In [8]:
# expand contractions: you're --> you are, i'm --> i am
question1 = df['question1']
question2 = df['question2']
for contraction, expanded in tqdm(contractions.contractions_dict.items(), desc="Expanding Contractions..."):
    question1 = question1.str.replace(rf'\b{re.escape(contraction.lower())}\b', expanded.lower(), regex=True)
    question2 = question2.str.replace(rf'\b{re.escape(contraction.lower())}\b', expanded.lower(), regex=True)
df['question1'] = question1
df['question2'] = question2

Expanding Contractions...: 100%|████████████████████████████████████████████████████| 344/344 [00:19<00:00, 17.84it/s]


In [9]:
# expand chatwords
with open("datasets/chat_words.yaml", "r", encoding="utf-8") as f:
    chat_words = yaml.safe_load(f)

question1 = df['question1']
question2 = df['question2']
for chat_word, full_word in tqdm(chat_words.items(), desc="Correcting Questions"):
    question1 = question1.str.replace(rf'\b{re.escape(chat_word.lower())}\b', full_word, regex=True)
    question2 = question2.str.replace(rf'\b{re.escape(chat_word.lower())}\b', full_word, regex=True)
df['question1'] = question1
df['question2'] = question2

Correcting Questions: 100%|█████████████████████████████████████████████████████████| 106/106 [00:04<00:00, 23.11it/s]


In [10]:
# remove stopwords
stop_words = stopwords.words('english')
stopword_removed_q1 = df['question1']
stopword_removed_q2 = df['question2']
for stop_word in tqdm(stop_words, desc='Removing Stop Words'):
    stopword_removed_q1 = stopword_removed_q1.str.replace(rf'\b{re.escape(stop_word)}\b', '', regex=True)
    stopword_removed_q2 = stopword_removed_q2.str.replace(rf'\b{re.escape(stop_word)}\b', '', regex=True)
df['question1'] = stopword_removed_q1
df['question2'] = stopword_removed_q2

Removing Stop Words: 100%|██████████████████████████████████████████████████████████| 198/198 [00:07<00:00, 25.13it/s]


In [11]:
# remove emojis
emoji_pattern = r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251]+'
df['question1'] = df['question1'].str.replace(emoji_pattern, '', regex=True)
df['question2'] = df['question2'].str.replace(emoji_pattern, '', regex=True)

In [12]:
# remove punctuations
df['question1'] = df['question1'].str.replace(r'''[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]''','',regex=True)
df['question2'] = df['question2'].str.replace(r'''[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]''','',regex=True)

In [13]:
# correct spellings using symspellpy
tokens_series1 = df['question1'].str.findall(r'\b[a-z]+\b')
tokens_series2 = df['question2'].str.findall(r'\b[a-z]+\b')
tokens_series = cudf.concat([tokens_series1,tokens_series2], axis=0, ignore_index=True) #tokenization per row

all_words = tokens_series.explode().dropna()   # all unique words
unique_words = all_words.unique().to_pandas()  # CPU transfer

sym = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dict_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym.load_dictionary(dict_path, term_index=0, count_index=1)

# creating spell correction map (dictionary) for current data
correction_map = {}
for word in unique_words:
    if word not in sym.words:  # word not in dictionary → likely misspelled
        suggestions = sym.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        if suggestions:
            correction_map[word] = suggestions[0].term

# correcting the spellings
corrected_q1 = df['question1']
corrected_q2 = df['question2']
for wrong, right in tqdm(correction_map.items(), desc="Correcting Questions"):
    corrected_q1 = corrected_q1.str.replace(rf'\b{re.escape(wrong)}\b', right, regex=True)
    corrected_q2 = corrected_q2.str.replace(rf'\b{re.escape(wrong)}\b', right, regex=True)
df['question1'] = corrected_q1
df['question2'] = corrected_q2

Correcting Questions: 100%|█████████████████████████████████████████████████████| 31936/31936 [19:45<00:00, 26.93it/s]


In [14]:
# remove extra white spaces
df['question1'] = df['question1'].str.strip().str.replace(r'\s+',' ',regex=True)
df['question2'] = df['question2'].str.strip().str.replace(r'\s+',' ',regex=True)

In [15]:
df.to_pandas().to_csv('datasets/Quora Questions Pair Dataset/train_preprocessed.csv', index=False)